In [2]:
pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 36.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 30.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
# -------- Settings --------
DATA_JOBS = "/workspaces/vector_db_benchmarking/data/jobs.text"
DATA_CANDIDATES = "/workspaces/vector_db_benchmarking/data/candidates.text"

MODELS = {
    "E5_BASE_V2": "intfloat/e5-base-v2",
    "GTE_BASE": "thenlper/gte-base",
    "BGE_BASE_V15": "BAAI/bge-base-en-v1.5",
    "MPNET_BASE_V2": "sentence-transformers/all-mpnet-base-v2"
}

DISTANCE_METRICS = ["cosine", "l2", "dot"]
TOP_K_DEFAULT = 5


In [4]:
# -------- Helper Functions --------
def load_data(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_model(model_name):
    print(f"Loading model: {model_name}...")
    return SentenceTransformer(model_name)

def compute_similarity(query_vec, corpus_vecs, metric="cosine"):
    if metric == "cosine":
        sims = cosine_similarity(query_vec, corpus_vecs)[0]
    elif metric == "l2":
        sims = -np.linalg.norm(corpus_vecs - query_vec, axis=1)
    elif metric == "dot":
        sims = np.dot(corpus_vecs, query_vec.squeeze())
    else:
        raise ValueError(f"Unknown metric: {metric}")
    return sims

def top_k_indices(scores, k):
    return np.argsort(scores)[::-1][:k]

In [5]:
#Loading Data:
jobs_texts = load_data(DATA_JOBS)
candidates_texts = load_data(DATA_CANDIDATES)

In [6]:
#Embedding Data:

embeddings_by_model = {}

for model_key, model_name in MODELS.items():
    model = load_model(model_name)
    jobs_embeds = model.encode(jobs_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=False)
    candidates_embeds = model.encode(candidates_texts, batch_size=32, show_progress_bar=True, normalize_embeddings=False)
    embeddings_by_model[model_key] = {
        "jobs": jobs_embeds,
        "candidates": candidates_embeds
    }

print("\n✅ All embeddings ready!\n")

Loading model: intfloat/e5-base-v2...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Loading model: thenlper/gte-base...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Loading model: sentence-transformers/all-mpnet-base-v2...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


✅ All embeddings ready!



In [15]:
query = "house wife"
top_k = 3
dataset_choice = "jobs"

for model_key, model_name in MODELS.items():
        print(f"\n\n====== MODEL: {model_key} ======\n")
        
        model = SentenceTransformer(model_name)
        query_vec = model.encode([query], normalize_embeddings=False)
        corpus_embeds = embeddings_by_model[model_key][dataset_choice]
        corpus_texts = jobs_texts if dataset_choice == "jobs" else candidates_texts

        for metric in DISTANCE_METRICS:
            print(f"\n--- Distance Metric: {metric.upper()} ---")
            
            start_time = time.time()
            scores = compute_similarity(query_vec, corpus_embeds, metric=metric)
            indices = top_k_indices(scores, top_k)
            elapsed = time.time() - start_time

            for rank, idx in enumerate(indices, 1):
                text_snippet = corpus_texts[idx][:400].strip().replace("\n", " ")
                print(f"Rank {rank} (Score: {scores[idx]:.4f}): {text_snippet}...")
            print(f"⏱️ Query Time: {elapsed:.2f} seconds")



====== MODEL: E5_BASE_V2 ======


--- Distance Metric: COSINE ---
Rank 1 (Score: 0.7335): Title: Luxury Goods Sales Manager. Company: Prestige Atelier. Location: Jeddah, Saudi Arabia. Sector: Retail. Job Type: Full-Time. Description: Manage the sales team and operations of a luxury goods boutique. Ensure exceptional customer service and achieve sales targets. Experience in luxury retail management is required.. Majors: Business Administration, Marketing, Luxury Brand Management. Educat...
Rank 2 (Score: 0.7317): Title: Sommelier. Company: Grandeur Hospitality. Location: Doha, Qatar. Sector: Hospitality. Job Type: Full-Time. Description: Manage wine cellars and provide expert wine service to guests in a high-end establishment. Develop wine lists and train staff on wine knowledge. Relevant certifications are highly valued.. Majors: Hospitality Management, Culinary Arts. Education: Certification (e.g., Court...
Rank 3 (Score: 0.7256): Title: Financial Trader. Company: Global Trade House